# Tune `mspc_rf`

MSPC + Random Forest. Repeated stratified CV GridSearchCV on the train split; 
writes [`data/processed/tuned/mspc_rf.json`](../data/processed/tuned/mspc_rf.json).

**Hyperparameters:** PLS `n_components`, classifier `max_depth`.

**Selection:** maximize mean ROC AUC across CV folds.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

# Repo root when kernel cwd is SECOM/ or SECOM/tuning/
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tuned_params_path,
)

MODEL_ID = "mspc_rf"
spec = MODEL_SPECS[MODEL_ID]


In [2]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [3]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_mspc__pls__n_components,classifier__max_depth
0,"[5, 10, 15, 20]",NaN
1,NaN,"[3, 4, 6]"


In [4]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


mspc_rf: 12 candidates x 25 folds = 300 fits


GridSearchCV 300 fits:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

Fitting 25 folds for each of 12 candidates, totalling 300 fits
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.2s
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.2s
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.2s
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.3s
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.2s
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.3s
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.3s
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.3s
[CV] END classifier__max_depth=3, preprocess__sensor_mspc__pls__n_components=5; total time=   3.3s
[CV] END classifier__max_depth=3, preprocess__

In [5]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
payload = save_tuned_params(spec, cv_summary, fold_results, aggregated)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
print("Best config (mean ROC AUC):")
display(aggregated.head(10))
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/mspc_rf.json
Best config (mean ROC AUC):


,n_components,max_depth,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
11,20,6,43.560080,3.779832,0.564399,17.529412,7.302563,95.350427,1.298020,0.711336,0.066982,0.186323,0.061602
10,20,4,42.595840,3.956937,0.574042,21.338235,7.890963,93.470085,1.407119,0.710831,0.065069,0.187273,0.061528
9,20,3,41.121858,4.615736,0.588781,26.132353,9.391016,91.623932,1.598999,0.709072,0.064534,0.186759,0.060822
7,15,4,41.236425,5.403788,0.587636,24.911765,10.802709,92.615385,1.573764,0.707134,0.067843,0.186698,0.061603
8,15,6,42.978695,5.267455,0.570213,19.632353,10.418368,94.410256,1.424661,0.706160,0.071508,0.189140,0.068210
6,15,3,40.961350,5.079345,0.590387,27.102941,10.569080,90.974359,1.663135,0.703737,0.067040,0.185191,0.063022
4,10,4,41.688223,5.982620,0.583118,24.897059,12.094424,91.726496,1.808576,0.698691,0.067027,0.184952,0.057381
3,10,3,40.600930,4.744427,0.593991,28.558824,9.762934,90.239316,1.729286,0.696773,0.065037,0.187613,0.059015
5,10,6,43.007039,5.210569,0.569930,20.823529,10.357410,93.162393,1.833129,0.694887,0.068149,0.182805,0.059764
1,5,4,40.505153,5.325121,0.594948,28.852941,10.289978,90.136752,1.996856,0.693110,0.070412,0.190089,0.063376


{'preprocess__sensor_mspc__pls__n_components': 20, 'classifier__max_depth': 6}